In [ ]:
# Daraz Scraper Project

import time
import csv
import re
from dataclasses import dataclass, asdict
from typing import List

In [ ]:
# External Libraries

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
# Product Model

@dataclass
class Product:
    title: str
    price: str
    discount: str
    reviews: str
    sold: str
    url: str
    image_url: str

In [ ]:
# Create and configure Chrome driver


def get_driver(headless=True):
    options = Options()

    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    # Helps avoid bot detection
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    return driver

In [ ]:
# Scroll page to load lazy content

def _scroll_page(driver):
    for i in range(4):
        driver.execute_script(f"window.scrollTo(0, {(i+1)*1000});")
        time.sleep(1.5)


In [ ]:
# Parse HTML and extract product data

def _parse_page(soup) -> List[Product]:
    products = []

    # Different selectors because Daraz UI changes frequently

    cards = soup.select("div[data-qa-locator='product-item'], .gridItem--Yd0sa, .box--uj89G")

    for card in cards:
        try:
            
            # TITLE + PRODUCT LINK
           
            title_el = card.select_one(".RfADt a, .title--w9WBG a, a[title]")
            title = title_el.get_text(strip=True) if title_el else "N/A"

            link = title_el.get("href", "") if title_el else ""
            if link and not link.startswith("http"):
                link = "https:" + link if link.startswith("//") else "https://www.daraz.pk" + link

            # ---------------------------------------------------------------------------------------------
            # PRICE
            
            price_el = card.select_one(".aBrP0, .price--u7Byf, [class*='price']")
            price = price_el.get_text(strip=True).split("Coins")[0].strip() if price_el else "N/A"

            # ---------------------------------------------------------------------------------------------
            # DISCOUNT
            
            discount = "N/A"
            disc_el = card.select_one(".IcOsH, .discount--m9kdH, [class*='discount'], del, .WNoq3")

            if disc_el:
                temp_disc = disc_el.get_text(strip=True)
                if any(x in temp_disc for x in ["%", "Off", "-"]):
                    discount = temp_disc

            # ---------------------------------------------------------------------------------------------
            # REVIEWS & SOLD ITEMS
            
            meta_text = card.get_text(" ", strip=True)

            reviews, sold = "0", "0"

            # Extract reviews like (123)
            rev_match = re.search(r"\((\d+)\)", meta_text)
            if rev_match:
                reviews = rev_match.group(1)

            # Extract sold like "50 sold" or "1.2K sold"
            sold_match = re.search(r"([\d\.]+K?)\s*sold", meta_text, re.IGNORECASE)
            if sold_match:
                sold = sold_match.group(1)

            # ---------------------------------------------------------------------------------------------
            # IMAGE URL
            
            img_el = card.select_one("img")
            image_url = "N/A"

            if img_el:
                image_url = (
                    img_el.get("data-src")
                    or img_el.get("src")
                    or img_el.get("lazy-src")
                    or "N/A"
                )

                if image_url.startswith("//"):
                    image_url = "https:" + image_url
            
            #---------------------------------------------------------------------------------------------
            # Save product object

            products.append(Product(
                title=title,
                price=price,
                discount=discount,
                reviews=reviews,
                sold=sold,
                url=link,
                image_url=image_url
            ))

        except Exception:
            continue

    return products

